# Testing image statistics using Carcará-X scans

In [ ]:
%env PYTHONPATH=/opt/micromamba/envs/sirius/repos/dev-packages:${PYTHONPATH}
import sys
sys.path.insert(0, "/opt/micromamba/envs/sirius/repos/dev-packages/siriuspy/")
sys.path.insert(1, "/home/ABTLUS/arnaldo.filho/repos/cax-scripts")

In [ ]:
from caxscripts.image_statistics import Histogram2DAnalyzer as H2DA
from caxscripts.scananalysis import data_from_h5_files, files_in_directory
import h5py
import numpy as np
import os
import re
import matplotlib.pyplot as plt

### Overview of the Analyzer class

In [ ]:
print(H2DA.__doc__)
print("#"*80)

### Files from the TX scans performed @ 2026-03-19

In [ ]:
# This way we can easily change the directory

workdir = "/home/gabriel.amici/testing_data/2026-03-19"
# workdir = "/home/arnaldo.filho/Carcara/measurement/2026-03-19"
file_pattern = r"mirror_tx_pass0*"


In [ ]:
files = files_in_directory(workdir, file_pattern)
data  = data_from_h5_files(files)

print(f"Loaded {len(files)} files matching pattern '{file_pattern}'"
      f" from '{workdir}'\n Files:")
for f in files:
    print(f" * {f}")


### Sample image

In [ ]:
test_img = data['01']['scan-0000']['dvf_B1']['data'].T

# Crude plot to check the image looks correct
plt.imshow(test_img.T, origin='lower')
plt.colorbar()
plt.title("Test Image")
plt.xlabel("Pixel X")
plt.ylabel("Pixel Y")
plt.show()

In [ ]:
# We can now create the analyzer and test it on the image

ana = H2DA(img=test_img,
           xedges=np.arange(test_img.shape[0]+1),
           yedges=np.arange(test_img.shape[1]+1))

There are two main methods for extracting the ditribution parameters
of the image: one is based on computing the moments of the distribution
and using them as the parameters for a biviriate normal distribution;
the other is to perform a nonlinear 2-dimensional gaussian fit over
the image, using the computed moments as an initial guess.

First, we shall look at the calculated moments and the resulting ellipse:

In [ ]:
hprm_mom = ana.compute_moments()
ana.print_stats(hprm_mom)

In [ ]:
hprm_mom

In [ ]:
hprm_qck = ana.hprm_qck
ana.print_stats(hprm_qck)

Now let's check the resulting ellipse:

In [ ]:
# Global limits for the plot
xlim = (ana.hprm_mom['mux'] - 3.5*ana.hprm_mom['sigx'],
        ana.hprm_mom['mux'] + 3.5*ana.hprm_mom['sigx'])
ylim = (ana.hprm_mom['muy'] - 3.5*ana.hprm_mom['sigy'],
        ana.hprm_mom['muy'] + 3.5*ana.hprm_mom['sigy'])

In [ ]:
fig, ax = ana.plot(hprm=hprm_mom)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_title("DVF B1 Image with Moments")
plt.show()


Background noise impedes the moments of this histogram of being an accurate description of the distribution. We can see the difference by fitting a Gaussian distribution using the moments as guesses:

In [ ]:
hprm_fit = ana.fit(hprm=hprm_mom, useroi=True)
ana.print_stats(hprm_fit)

The fitted ellipse should look more adjusted to the image:

In [ ]:
fig, ax = ana.plot(hprm=hprm_fit)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_title("DVF B1 Image with fitted Ellipse")
plt.show()

The improvement is evident, at the cost of computation time (which can be improved by first defining a ROI inside of which the fitting shall take place).

Now we shall analyze the impact of the background noise by first computing the Kapur entropy threshold:

In [ ]:
entropies, bin_edges, optimal_threshold, _ = ana.compute_threshold()
ana.plot_entropy(entropies, bin_edges, optimal_threshold)
plt.show()

The use of `compute_threshold` yelds a new class attribute named `image_thresholded`, which we can plot to check the discarded area:

In [ ]:
# We can also plot the image with the optimal threshold
plt.imshow(ana.img_thresholded.T, origin='lower')
plt.colorbar()
plt.xlabel("Pixel X")
plt.ylabel("Pixel Y")
plt.xlim(xlim)
plt.ylim(ylim)
plt.title("DVF B1 Image with Optimal Threshold")
plt.show()

Now we can define a new Analyzer whose image is just the foreground of `ana_scan.img`

In [ ]:
anath = H2DA(img=ana.img_thresholded,
             xedges=ana.xedges,
             yedges=ana.yedges)

In [ ]:
thrs_hprm_mom = anath.compute_moments()
anath.print_stats(thrs_hprm_mom)

Comparing with the fitted parameters of `ana_scan`:

In [ ]:
ana.print_stats(hprm_fit)

In [ ]:
fig, ax = anath.plot(hprm=thrs_hprm_mom)
ax.set_title("DVF B1 Image masked and with Moments")
plt.xlim(xlim)
plt.ylim(ylim)
plt.show()

The result is more similar to the fitted parameters of the original `ana_scan`, but sigmas are subestimated: threshold value seems to be high, eliminating parts of the light halo around the beam image.

Finally, we can also fit a 2D Gaussian over the thresholded image:

In [ ]:
thrs_hprm_fit = anath.fit(hprm=thrs_hprm_mom)
anath.print_stats(thrs_hprm_fit)

This is still underestimated... the plot is informative:

In [ ]:
fig, ax = anath.plot(hprm=thrs_hprm_fit)
ax.set_title("DVF B1 Image masked and with fitted Ellipse")
plt.xlim(xlim)
plt.ylim(ylim)
plt.show()

To better visualize the differences, we can make a general plot:

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(16, 12))

ana.plot(hprm=hprm_mom, fig=fig, ax=axs[0, 0])
axs[0, 0].set_title("DVF B1 Image with Moments")

ana.plot(hprm=hprm_fit, fig=fig, ax=axs[0, 1])
axs[0, 1].set_title("DVF B1 Image with fitted Ellipse")

anath.plot(hprm=thrs_hprm_mom, fig=fig, ax=axs[1, 0])
axs[1, 0].set_title("DVF B1 Image masked and with Moments")

anath.plot(hprm=thrs_hprm_fit, fig=fig, ax=axs[1, 1])
axs[1, 1].set_title("DVF B1 Image masked and with fitted Ellipse")

for i in range(2):
    for j in range(2):
        axs[i, j].set_xlim(xlim)
        axs[i, j].set_ylim(ylim)

plt.tight_layout()
plt.show()

#### Plot images and their projections.

Function to plot image and its projections.

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable


def plot_projections(ana, img, hprm, xlim, ylim, title=None):
    """Plot the image with the fitted ellipse and respective projections.

    Arguments:
        ana  : Histogram2DAnalyzer instance with the image and parameters.
        img  : image data to plot.
        hprm : dictionary of parameters to plot the ellipse.
        xlim : horizontal limits for the main image plot.
        ylim : vertical limits for the main image plot.
        title : title for the main image plot.
    """
    fig, ax0 = plt.subplots(figsize=(14, 6))

    divider = make_axes_locatable(ax0)
    axb = divider.append_axes("bottom", size="25%", pad=0.3, sharex=ax0)
    axr = divider.append_axes("right", size="25%", pad=0.3, sharey=ax0)

    # Plot the main image with the ellipses (no colorbar to avoid displacement).
    ana.plot(hprm=hprm, fig=fig, ax=ax0, colorbar=False)
    ax0.set_xlim(xlim)
    ax0.set_ylim(ylim)
    if title is not None:
        ax0.set_title(title)

    # Suppress redundant tick labels
    ax0.tick_params(labelbottom=False)
    axr.tick_params(labelleft=False)

    xproj = np.sum(img, axis=0)
    py = np.arange(len(xproj))
    axr.plot(xproj, py, drawstyle='steps-post')

    # axr.axhline(hprm['muy'], color='red', linestyle='--', label='Center Y')

    yproj = np.sum(img, axis=1)
    axb.plot(yproj, drawstyle='steps-post')

    axb.grid(True)
    axr.grid(True)


In [ ]:
title = "Original image and sigma-ellipses from moments"
plot_projections(ana, ana.img, hprm_mom, xlim, ylim, title=title)

title = "Original image and sigma-ellipses from fit"
plot_projections(ana, ana.img, hprm_fit, xlim, ylim, title=title)

title = "Thresholded image and sigma-ellipses from moments"
plot_projections(anath, anath.img, thrs_hprm_mom, xlim, ylim, title=title)

title = "Thresholded image and sigma-ellipses from fit"
plot_projections(anath, anath.img, thrs_hprm_fit, xlim, ylim, title=title)

plt.show()


And we can overview all the parameters:

In [ ]:
# The table of results can be printed using the tabulate library for better formatting,
# install it if you don't have it already:

%pip install tabulate

In [ ]:
thrs_hprm_qck = anath.hprm_qck

In [ ]:
from tabulate import tabulate

# Collect all analysis results
results = [
    ["Quick",
     hprm_qck['mux'],
     hprm_qck['muy'],
     hprm_qck['sigx'],
     hprm_qck['sigy'],
     0,
     0],
    ["Moments",
     hprm_mom['mux'],
     hprm_mom['muy'],
     hprm_mom['sig_major'],
     hprm_mom['sig_minor'],
     hprm_mom['theta'],
     hprm_mom['cov'][0, 1]],
    ["Fitted Ellipse",
     hprm_fit['mux'],
     hprm_fit['muy'],
     hprm_fit['sig_major'],
     hprm_fit['sig_minor'],
     hprm_fit['theta'],
     hprm_fit['cov'][0, 1]],
    ["Quick Thresholded",
     thrs_hprm_qck['mux'],
     thrs_hprm_qck['muy'],
     thrs_hprm_qck['sigx'],
     thrs_hprm_qck['sigy'],
     0,
     0],
    ["Thresholded Moments",
     thrs_hprm_mom['mux'],
     thrs_hprm_mom['muy'],
     thrs_hprm_mom['sig_major'],
     thrs_hprm_mom['sig_minor'],
     thrs_hprm_mom['theta'],
     thrs_hprm_mom['cov'][0, 1]],
    ["Thresholded Fitted Ellipse",
     thrs_hprm_fit['mux'],
     thrs_hprm_fit['muy'],
     thrs_hprm_fit['sig_major'],
     thrs_hprm_fit['sig_minor'],
     thrs_hprm_fit['theta'],
     thrs_hprm_fit['cov'][0, 1]],
]

headers = [
    "Method",
    "Mux",
    "Muy",
    "Sigma Major",
    "Sigma Minor",
    "Theta (rad)",
    "Cov(X,Y)"
    ]

# Format floats to 2 decimal places
table_data = [
    [row[0]] + [f"{val:.2f}" for val in row[1:]]
    for row in results
    ]

print("Summary of statistics:")
print("\n" + tabulate(table_data, headers=headers, tablefmt="grid"))